# BiVLA — SpatialVLA Latent-Focus on SimplerEnv (Colab Pro)

멘토님의 **BiVLA 모노레포**(`SoumyaratnaDebnath/BiVLA`)에서 SpatialVLA latent-focus 경로를
SimplerEnv(WidowX Bridge)로 돌리는 노트북.

기존 `trillion-boy/spatialvla` 단일 repo용 셋업을 BiVLA 모노레포에 맞춰 포팅했다.

### 이 버전의 핵심 차이 (꼭 읽기)
- **클론 대상이 BiVLA 모노레포**다. `latent_saccade_spatialvla.py`가 `../../..`(=BiVLA 루트)를
  자동으로 sys.path에 넣어 `shared_unified_policy`를 찾으므로, 디렉터리 구조만 유지하면 import는 자동 해결된다.
- **shared controller가 작업별 설정을 자동 적용**한다. `step()`에서 instruction을 인식하면
  `_apply_task_policy_profile()`이 CLI 가중치(`--grasp-fovea-weight` 등)를 **archetype 프로파일로 덮어쓴다.**
  → 따라서 run 명령에 가중치 플래그를 줄 필요가 없다. **task만 주면 된다.**
- 멘토님 최종 컨트롤러는 **선택적(selective)**: `stack_alignment`에서만 focus ON,
  carrot/eggplant/spoon은 baseline. → **ON vs OFF 결과는 stack에서만 달라진다.**
  (carrot ON ≈ carrot OFF 가 정상이다. 헷갈리지 말 것.)

> `%%bash` 셀은 bash, 나머지는 Python 셀이다. Colab Pro(A100/L4) 권장.


## 1. Miniconda 설치

In [ ]:
%%bash
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
bash /tmp/miniconda.sh -b -f -p /usr/local
conda --version

## 2. conda TOS 동의

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

## 3. conda 환경 생성 (spatialvla, python 3.10)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda create -n spatialvla python=3.10.12 -y
conda run -n spatialvla python --version

## 4. 시스템 렌더링 의존성 (EGL / Vulkan / ffmpeg)

In [ ]:
%%bash
apt-get update -yqq
apt-get -yqq install libegl1-mesa libegl1 libgl1 libosmesa6-dev
apt-get install -yqq --no-install-recommends libvulkan-dev vulkan-tools
apt-get install -yqq ffmpeg

## 5. 컴파일러 + 기본 파이썬 패키지 (transformers 4.47.0 필수)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n spatialvla conda install -c conda-forge gcc=12.1.0 gxx_linux-64 -y
conda run -n spatialvla pip install mediapy
# SpatialVLA는 PaliGemma2 백본 → transformers 4.47.0 필수
conda run -n spatialvla pip install transformers==4.47.0 tokenizers==0.21.0 pillow
conda run -n spatialvla pip install matplotlib

## 6. BiVLA 모노레포 + SimplerEnv 클론 및 시뮬 의존성

**변경점:** 기존 `trillion-boy/spatialvla` 대신 **멘토님 BiVLA 모노레포**를 클론한다.
SpatialVLA 정책은 번들 `BiVLA/SimplerEnv`에 없으므로 `DelinQu/SimplerEnv-OpenVLA` fork를 따로 클론한다.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh

# ── 멘토님 BiVLA 모노레포 (UniVLA + SpatialVLA + AutoGaze + shared controller) ──
if [ ! -d /content/BiVLA ]; then
  git clone --depth 1 https://github.com/SoumyaratnaDebnath/BiVLA.git /content/BiVLA
fi

# ── SimplerEnv (SpatialVLA/OpenVLA 지원 fork) ──
if [ ! -d /content/SimplerEnv ]; then
  git clone https://github.com/DelinQu/SimplerEnv-OpenVLA \
    --recurse-submodules -q /content/SimplerEnv
fi

# SAPIEN 2.2.2
conda run -n spatialvla pip install sapien==2.2.2

# ManiSkill2_real2sim
conda run -n spatialvla pip install --no-deps -e /content/SimplerEnv/ManiSkill2_real2sim

# gym
conda run -n spatialvla conda install -c conda-forge gym=0.21.0 -y

# ruckig
conda run -n spatialvla pip install ruckig

# opencv + 기타 시뮬 의존성
conda run -n spatialvla pip install \
  transforms3d "opencv-python-headless==4.8.1.78" \
  "trimesh==3.22.5" "open3d==0.17.0" "mplib==0.0.9" "gymnasium==0.29.1"

# mani-skill2 누락 deps
conda run -n spatialvla pip install \
  gdown GitPython h5py \
  imageio "imageio[ffmpeg]" \
  rtree tabulate

# numpy 고정 (SAPIEN/ManiSkill 호환)
conda run -n spatialvla pip install "numpy==1.24.4"

# SimplerEnv
conda run -n spatialvla pip install --no-deps -e /content/SimplerEnv

## 7. opencv / numpy 버전 재고정

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n spatialvla pip install "opencv-python==4.8.1.78"
conda run -n spatialvla pip install "numpy==1.24.4" 

## 8. xvfb (헤드리스 디스플레이)

In [ ]:
%%bash
apt-get install -yqq xvfb
echo "xvfb done" 

## 9. Vulkan ICD 설정

In [ ]:
%%bash
mkdir -p /etc/vulkan/icd.d
cat > /etc/vulkan/icd.d/nvidia_icd.json << 'EOF'
{
    "file_format_version": "1.0.0",
    "ICD": {
        "library_path": "libGLX_nvidia.so.0",
        "api_version": "1.2.155"
    }
}
EOF
mkdir -p /usr/share/vulkan/implicit_layer.d
echo "Vulkan config done" 

## 10. SAPIEN renderer_config 확인 (필요 시)

In [ ]:
import os, sys

rc_path = None
search_dirs = ["/usr/local/envs/spatialvla/lib/python3.10/site-packages"]
for sp in search_dirs:
    p = os.path.join(sp, 'sapien', 'core', 'renderer_config.py')
    if os.path.exists(p):
        rc_path = p
        break

if rc_path is None:
    print("WARN: renderer_config.py 못 찾음 (SAPIEN 버전에 따라 없을 수 있음 — 건너뜀)")
else:
    print(f"패치 대상: {rc_path}")
    with open(rc_path, 'r') as f:
        content = f.read()
    print("=== 원본 (앞 300자) ===")
    print(content[:300])

## 11. pkg_resources shim (setuptools 충돌 회피)

In [ ]:
import os

site_packages = "/usr/local/envs/spatialvla/lib/python3.10/site-packages"
pr_dir = os.path.join(site_packages, "pkg_resources")

if os.path.exists(os.path.join(pr_dir, "__init__.py")):
    try:
        import importlib, subprocess
        out = subprocess.run(
            ["/usr/local/envs/spatialvla/bin/python", "-c", "import pkg_resources; print('ok')"],
            capture_output=True, text=True
        )
        if "ok" in out.stdout:
            print("pkg_resources 정상 작동 — shim 불필요")
            raise SystemExit
    except SystemExit:
        pass

os.makedirs(pr_dir, exist_ok=True)
shim = '''"""pkg_resources shim — importlib.metadata 기반"""
import importlib
import importlib.metadata as _meta
import os

def get_distribution(name):
    class _Dist:
        try:
            version = _meta.version(name)
        except Exception:
            version = "0.0.0"
    return _Dist()

def require(requirements):
    pass

def resource_filename(package_name, resource_name):
    try:
        mod = importlib.import_module(package_name)
        return os.path.join(os.path.dirname(mod.__file__), resource_name)
    except Exception:
        return resource_name

def resource_string(package_name, resource_name):
    path = resource_filename(package_name, resource_name)
    with open(path, "rb") as f:
        return f.read()

def resource_exists(package_name, resource_name):
    path = resource_filename(package_name, resource_name)
    return os.path.exists(path)

def resource_stream(package_name, resource_name):
    path = resource_filename(package_name, resource_name)
    return open(path, "rb")

class WorkingSet:
    def __iter__(self): return iter([])
    def __contains__(self, item): return False
    def require(self, *a, **kw): pass

working_set = WorkingSet()
'''
with open(os.path.join(pr_dir, "__init__.py"), "w") as f:
    f.write(shim)
print(f"pkg_resources shim 생성: {pr_dir}/__init__.py")

## 12. 시뮬레이터 import 검증

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n spatialvla python -c "import pkg_resources; print('pkg_resources OK')"
conda run -n spatialvla python -c "import sapien.core; print('sapien OK')"
conda run -n spatialvla python -c "import simpler_env; print('simpler_env OK')" 

## 13. GroundingDINO 의존성 확인 (Latent Focus의 bbox 탐지)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n spatialvla pip install torchvision --quiet
conda run -n spatialvla python -c "
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torchvision
print('transformers DINO 지원: OK')
print('torchvision:', torchvision.__version__)
"

## 14. SpatialVLA 모델 다운로드 (HF hub, ~8.5GB)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n spatialvla --no-capture-output python -c "
from huggingface_hub import snapshot_download
print('Downloading SpatialVLA spatialvla-4b-224-pt ...')
snapshot_download(
    repo_id='IPEC-COMMUNITY/spatialvla-4b-224-pt',
    local_dir='/content/pretrain/spatialvla-4b-224-pt',
    ignore_patterns=['*.git*', '*.bin'],
)
print('Done: SpatialVLA weights')
"

## 15. 경로 검증

In [ ]:
import os
model_path = "/content/pretrain/spatialvla-4b-224-pt"
print(f"{'OK' if os.path.isdir(model_path) else 'MISSING'} SpatialVLA model: {model_path}")
if os.path.isdir(model_path):
    files = sorted(os.listdir(model_path))
    print("   files:", files[:20])
    for need in ["config.json", "modeling_spatialvla.py", "processing_spatialvla.py"]:
        mark = "OK" if need in files else "MISSING"
        print(f"   {mark}: {need}")

## 16. torch CUDA 12.1 재설치 (Colab GPU 매칭)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda run -n spatialvla pip install \
  torch==2.5.1 torchvision==0.20.1 \
  --index-url https://download.pytorch.org/whl/cu121 -q
conda run -n spatialvla python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 17. BiVLA import 동작 검증 (모노레포 핵심 확인)

`latent_saccade_spatialvla.py`가 BiVLA 루트의 `shared_unified_policy`를 자동으로 찾는지 확인.
여기서 OK가 떠야 run이 정상 동작한다.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
export PYTHONPATH=/content/BiVLA:/content/BiVLA/SpatialVLA:$PYTHONPATH
cd /content/BiVLA
conda run -n spatialvla python -c "
import sys
sys.path.insert(0, '/content/BiVLA')
sys.path.insert(0, '/content/BiVLA/SpatialVLA')
from shared_unified_policy import shared_task_policy_profile, infer_task_archetype
for instr in ['put the carrot on the plate', 'stack the green block on the yellow block',
              'put eggplant into basket', 'put spoon on towel']:
    p = shared_task_policy_profile(instr, model_family='spatialvla')
    print(f'{infer_task_archetype(instr):18s} focus_enabled={p.spatial_focus_enabled}  gate={p.spatial_phase_gate}')
print('shared_unified_policy import: OK')
"

## 18. 평가 실행 스크립트 작성

`/content/run_eval.sh <task> <on|off> <output_dir>` 형태로 재사용.
- BiVLA 모노레포 경로(`SpatialVLA/experiments/latent_saccade/spatialvla_eval.py`)로 실행
- `PYTHONPATH`에 BiVLA 루트 + SpatialVLA 포함
- ON = shared controller 자동 적용(가중치 플래그 불필요), OFF = `--no-latent-mask`
- `N_EPISODES` 환경변수로 에피소드 수 조절 (스모크 테스트용)

In [ ]:
%%writefile /content/run_eval.sh
#!/bin/bash
set -e
TASK="$1"; MODE="$2"; OUTDIR="$3"

export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
export SIMPLER_ENV_ROOT=/content/SimplerEnv
export PYTHONPATH=/content/BiVLA:/content/BiVLA/SpatialVLA:$PYTHONPATH

# headless display
Xvfb :99 -screen 0 1280x1024x24 >/dev/null 2>&1 &
XVFB_PID=$!
sleep 2
export DISPLAY=:99

EXTRA=""
if [ "$MODE" = "off" ]; then EXTRA="--no-latent-mask"; fi

cd /content/BiVLA
/usr/local/envs/spatialvla/bin/python \
  SpatialVLA/experiments/latent_saccade/spatialvla_eval.py \
    --model-path /content/pretrain/spatialvla-4b-224-pt \
    --unnorm-key bridge_orig/1.0.0 \
    --task "$TASK" \
    --n-episodes "${N_EPISODES:-24}" \
    --output-dir "$OUTDIR" \
    $EXTRA \
    --save-video

kill $XVFB_PID 2>/dev/null || true

## 19. 스모크 테스트 (stack, ON, 4 에피소드)

먼저 4 에피소드로 파이프라인이 끝까지 도는지 확인. 로그에
`[LatentSaccade] shared profile archetype=stack_alignment focus=always ...` 가 보이면 컨트롤러 정상 동작.

In [ ]:
!N_EPISODES=4 bash /content/run_eval.sh widowx_stack_cube on /content/results/smoke_stack_on

## 20. 전체 평가 — Latent Focus ON (shared controller)

4개 task 전부 24 에피소드. shared controller가 stack만 focus를 켜고 나머지는 baseline으로 둔다.
(A100/L4 기준 task당 대략 수십 분. 세션이 끊기면 task별로 나눠 실행.)

In [ ]:
import subprocess
TASKS = [
    "widowx_put_eggplant_in_basket",
    "widowx_spoon_on_towel",
    "widowx_carrot_on_plate",
    "widowx_stack_cube",
]
for t in TASKS:
    print(f"\n===== ON: {t} =====", flush=True)
    subprocess.run(
        ["bash", "/content/run_eval.sh", t, "on", f"/content/results/{t}_on"],
        env={**__import__("os").environ, "N_EPISODES": "24"},
        check=False,
    )

## 21. 베이스라인 — Latent Focus OFF (대조군)

`--no-latent-mask`. 동일 코드로 hook만 비활성화되어 공식 SpatialVLA와 동일하게 동작한다.
(carrot/eggplant/spoon은 ON에서도 focus가 꺼져 있으므로 OFF와 사실상 동일 — stack 비교가 핵심.)

In [ ]:
import subprocess
TASKS = [
    "widowx_put_eggplant_in_basket",
    "widowx_spoon_on_towel",
    "widowx_carrot_on_plate",
    "widowx_stack_cube",
]
for t in TASKS:
    print(f"\n===== OFF: {t} =====", flush=True)
    subprocess.run(
        ["bash", "/content/run_eval.sh", t, "off", f"/content/results/{t}_off"],
        env={**__import__("os").environ, "N_EPISODES": "24"},
        check=False,
    )

## 22. 전체 결과 비교 (4개 task, ON vs OFF)

In [ ]:
import json, os

RESULTS = [
    ("PutEggplant->Basket", "widowx_put_eggplant_in_basket"),
    ("PutSpoon->Towel",      "widowx_spoon_on_towel"),
    ("PutCarrot->Plate",     "widowx_carrot_on_plate"),
    ("StackGreen->Yellow",   "widowx_stack_cube"),
]

def load(dir_path, task_key):
    fp = os.path.join(dir_path, f"results_{task_key}.json")
    if not os.path.exists(fp):
        return None
    with open(fp) as f:
        return json.load(f)

print(f"{'Task':<22} {'OFF grasp':>9} {'OFF succ':>9}  {'ON grasp':>9} {'ON succ':>9}  {'d succ':>7}")
print("-" * 74)
for label, task_key in RESULTS:
    off = load(f"/content/results/{task_key}_off", task_key)
    on  = load(f"/content/results/{task_key}_on",  task_key)
    off_g = f"{off['grasp_rate']:.1%}"   if off else "N/A"
    off_s = f"{off['success_rate']:.1%}" if off else "N/A"
    on_g  = f"{on['grasp_rate']:.1%}"    if on  else "N/A"
    on_s  = f"{on['success_rate']:.1%}"  if on  else "N/A"
    delta = f"{on['success_rate']-off['success_rate']:+.1%}" if (off and on) else ""
    print(f"{label:<22} {off_g:>9} {off_s:>9}  {on_g:>9} {on_s:>9}  {delta:>7}")
print("-" * 74)